# MCP Client with Claude: End-to-End Integration

## Table of contents
- [Architecture](#architecture)
- [Setup](#setup)
- [Part 1: Build and connect](#part-1-build-and-connect)
- [Part 2: Full agentic loop](#part-2-full-agentic-loop)
- [Part 3: MCP error handling](#part-3-mcp-error-handling)
- [Part 4: Production authentication](#part-4-production-authentication)
- [Exam practice](#exam-practice)
- [Quick reference](#quick-reference)

## Architecture

This notebook wires Claude's `tool_use` mechanism to an MCP server through an MCP client.

```
User message
    |
    v
Claude API (messages.create)   <-- claudeTools (converted from MCP schemas)
    |
    | tool_use block { name, input }
    v
Your code (agentic loop)
    |
    | callTool({ name, arguments })
    v
MCP Client  <--InMemoryTransport-->  MCP Server
                                          |
                                          | ListTools / CallTool handlers
                                          v
                                     Tool execution -> { content, isError? }
    |
    | tool_result { content, is_error? }
    v
Claude API  ->  final answer
```

**Key bridge**: MCP tools are discovered via `listTools()`, converted to
Claude's `input_schema` format, then Claude's `tool_use` requests are routed
through `callTool()` back to the MCP server. The MCP layer is invisible to Claude.

## Setup

This notebook requires `@modelcontextprotocol/sdk` (same as `02_mcp_server_basics`).
If you have not run notebook 02 yet, install it first:

```
deno add npm:@modelcontextprotocol/sdk
```

Also ensure `ANTHROPIC_API_KEY` is in `../.env`.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';
import { Server } from 'npm:@modelcontextprotocol/sdk/server/index.js';
import { Client } from 'npm:@modelcontextprotocol/sdk/client/index.js';
import { InMemoryTransport } from 'npm:@modelcontextprotocol/sdk/inMemory.js';
import {
  ListToolsRequestSchema,
  CallToolRequestSchema,
} from 'npm:@modelcontextprotocol/sdk/types.js';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const anthropic = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });

console.log('Imports ready. Model:', MODEL_NAME);

## Part 1: Build and connect

Build a minimal MCP server, connect an MCP client to it, discover its tools,
and convert the schemas to Claude's `input_schema` format.

In [ ]:
// MCP server with one Tool — includes both success and error path
const server = new Server(
  { name: 'portfolio-server', version: '1.0.0' },
  { capabilities: { tools: {} } },
);

server.setRequestHandler(ListToolsRequestSchema, async () => ({
  tools: [{
    name: 'get_portfolio_return',
    description:
      'Get the portfolio return percentage for a given quarter. ' +
      'Use when the user asks about portfolio performance or returns.',
    inputSchema: {
      type: 'object' as const,
      properties: {
        quarter: { type: 'string', description: 'Quarter, e.g. "Q1 2026"' },
      },
      required: ['quarter'],
    },
  }],
}));

server.setRequestHandler(CallToolRequestSchema, async (request) => {
  const { name, arguments: args } = request.params;
  const { quarter } = args as { quarter: string };

  if (name !== 'get_portfolio_return') {
    return {
      content: [{ type: 'text' as const, text: `Unknown tool: ${name}` }],
      isError: true,
    };
  }

  const returns: Record<string, number> = {
    'Q1 2026': 8.3,
    'Q4 2025': 12.1,
    'Q3 2025': -2.4,
  };

  const value = returns[quarter];
  if (value === undefined) {
    // Returns isError: true — client will see mcpResult.isError === true
    return {
      content: [{ type: 'text' as const, text: `No data available for ${quarter}` }],
      isError: true,
    };
  }

  return {
    content: [{ type: 'text' as const, text: `${quarter} portfolio return: ${value}%` }],
  };
});

console.log('MCP server configured with: get_portfolio_return');

In [ ]:
// Connect server and client via InMemoryTransport (no subprocess, no network)
const [clientTransport, serverTransport] = InMemoryTransport.createLinkedPair();
await server.connect(serverTransport);

const mcpClient = new Client(
  { name: 'claude-mcp-client', version: '1.0.0' },
  { capabilities: {} },
);
await mcpClient.connect(clientTransport);

// Discover tools from the MCP server
const { tools: mcpTools } = await mcpClient.listTools();
console.log('Discovered MCP tools:', mcpTools.map(t => t.name));

// Helper interface for converting MCP tool schemas
interface McpToolDef {
  name: string;
  description?: string;
  inputSchema?: { properties?: Record<string, unknown>; required?: string[] };
}

// Convert MCP Tool schema -> Claude input_schema format
function mcpToClaudeTool(t: McpToolDef): Anthropic.Tool {
  return {
    name: t.name,
    description: t.description ?? '',
    input_schema: {
      type: 'object',
      properties: (t.inputSchema?.properties ?? {}) as Record<string, unknown>,
      required: t.inputSchema?.required ?? [],
    } as Anthropic.Tool['input_schema'],
  };
}

const claudeTools = mcpTools.map(mcpToClaudeTool);

console.log('\nClaude-format tool (input_schema):');
console.log(JSON.stringify(claudeTools[0].input_schema, null, 2));

## Part 2: Full agentic loop

The loop does three things:
1. Calls `messages.create` with `claudeTools`
2. On `tool_use`: routes the call through `mcpClient.callTool()`
3. Checks `mcpResult.isError` before sending back a `tool_result`

This is the standard Claude + MCP integration pattern.

In [ ]:
async function askWithMcp(question: string): Promise<string> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: question }];

  while (true) {
    const response = await anthropic.messages.create({
      model: MODEL_NAME,
      max_tokens: 512,
      tools: claudeTools,
      messages,
    });

    console.log('  stop_reason:', response.stop_reason);

    if (response.stop_reason === 'end_turn') {
      const text = response.content.find(b => b.type === 'text');
      return text?.type === 'text' ? text.text : '(no text)';
    }

    messages.push({ role: 'assistant', content: response.content });

    const toolResults: Anthropic.ToolResultBlockParam[] = [];

    for (const block of response.content) {
      if (block.type !== 'tool_use') continue;

      console.log(`  -> tool_use: ${block.name}`, JSON.stringify(block.input));

      // Route through MCP client — this calls the server's CallTool handler
      const mcpResult = await mcpClient.callTool({
        name: block.name,
        arguments: block.input as Record<string, unknown>,
      });

      console.log(`  <- isError: ${mcpResult.isError ?? false}`);

      // Extract text from MCP content array
      const content = mcpResult.content
        .filter(c => c.type === 'text')
        .map(c => (c as { type: 'text'; text: string }).text)
        .join('\n');

      // Pass isError back to Claude via tool_result.is_error
      toolResults.push({
        type: 'tool_result',
        tool_use_id: block.id,
        content: content || '(no content)',
        ...(mcpResult.isError ? { is_error: true } : {}),
      });
    }

    messages.push({ role: 'user', content: toolResults });
  }
}

// Happy path: Q1 2026 has data
console.log('=== Query: Q1 2026 (success path) ===');
const answer = await askWithMcp('What was the portfolio return for Q1 2026?');
console.log('\nAnswer:', answer);

## Part 3: MCP error handling

When an MCP Tool call fails, the server returns `isError: true` in the result.

**MCP error response format:**

```json
{
  "isError": true,
  "content": [{ "type": "text", "text": "No data available for Q2 2024" }]
}
```

**With optional production metadata** (recognize these fields):

```json
{
  "isError": true,
  "isRetryable": false,
  "errorCategory": "NOT_FOUND",
  "content": [{ "type": "text", "text": "No data available for Q2 2024" }]
}
```

| Field | Purpose |
|---|---|
| `isError` | Marks the result as an error — always check this before using `content` |
| `isRetryable` | `false` = same call will always fail; `true` = transient error, retry may succeed |
| `errorCategory` | Category for routing (`NOT_FOUND`, `PERMISSION_DENIED`, `INTERNAL_ERROR`, etc.) |

The agentic loop from Part 2 already handles this: it passes `is_error: true` in the
`tool_result` so Claude knows the call failed and can respond accordingly.

In [ ]:
// Demonstrate the error response fields
const exampleErrorResponse = {
  isError: true,
  content: [{ type: 'text', text: 'No data available for Q2 2024' }],
};

const exampleErrorWithMeta = {
  isError: true,
  isRetryable: false,        // same call will always fail — do not retry
  errorCategory: 'NOT_FOUND',
  content: [{ type: 'text', text: 'No data available for Q2 2024' }],
};

console.log('Basic error response:');
console.log(JSON.stringify(exampleErrorResponse, null, 2));
console.log('\nWith production metadata:');
console.log(JSON.stringify(exampleErrorWithMeta, null, 2));

// Error path: Q2 2024 has no data — server returns isError: true
console.log('\n=== Query: Q2 2024 (error path) ===');
const errorAnswer = await askWithMcp('What was the portfolio return for Q2 2024?');
console.log('\nAnswer (error recovery):', errorAnswer);

## Part 4: Production authentication

MCP servers must read credentials from the environment, never from client request arguments.

**Rule**: Secrets flow in at server startup (env vars), not in per-request arguments.

In [ ]:
// Production authentication patterns

// Pattern 1: stdio server — inject credentials via "env" block in .mcp.json
const stdioAuthConfig = {
  mcpServers: {
    'data-server': {
      command: 'node',
      args: ['server.js'],
      transport: 'stdio',
      env: {
        DATABASE_URL: '${DATABASE_URL}',  // inherited from shell environment
        API_KEY: '${API_KEY}',            // never hardcode secrets here
      },
    },
  },
};

// Pattern 2: SSE server — credentials in Authorization header
const sseAuthConfig = {
  mcpServers: {
    'remote-server': {
      url: 'https://api.example.com/mcp',
      transport: 'sse',
      headers: {
        Authorization: 'Bearer ${MCP_TOKEN}',
      },
    },
  },
};

console.log('stdio auth config:');
console.log(JSON.stringify(stdioAuthConfig, null, 2));
console.log('\nSSE auth config:');
console.log(JSON.stringify(sseAuthConfig, null, 2));

// Pattern 3: server-side rule (code comment style)
console.log('\n--- Server-side auth rule ---');
console.log('');
console.log('  // In your MCP server handler:');
console.log('  const apiKey = Deno.env.get("API_KEY");   // ✅ read from env at startup');
console.log('  if (!apiKey) throw new Error("API_KEY not configured");');
console.log('');
console.log('  // NEVER do this:');
console.log('  // const apiKey = request.params.arguments.apiKey;  // ❌ secret from client');

## Exam practice

| Scenario | Answer |
|---|---|
| Claude outputs a `tool_use` block; how do you route it to an MCP server? | Call `mcpClient.callTool({ name, arguments })` |
| MCP server returns `{ isError: true, ... }`; what do you add to `tool_result`? | `is_error: true` — tells Claude the call failed |
| `isRetryable: false` on an error response — what does this mean? | Retrying the same call will always fail (e.g., permission denied, not a network error) |
| `errorCategory: "PERMISSION_DENIED"` vs `"NOT_FOUND"` — how should Claude respond differently? | PERMISSION_DENIED → stop and report; NOT_FOUND → may suggest alternative data |
| Where should an MCP server read API keys? | From environment variables (`Deno.env.get`), not from request arguments |
| Which transport allows multiple clients to connect simultaneously? | SSE (Streamable HTTP) |
| MCP tools still use which Claude API mechanism under the hood? | `tool_use` / `tool_result` — MCP is a layer above the API |

## Quick reference

### Core bridge: MCP tool -> Claude tool_use

```
1. mcpClient.listTools()          -> discover tool schemas from server
2. mcpToClaudeTool(mcpTool)       -> convert inputSchema to input_schema
3. messages.create({ tools })     -> Claude may output tool_use block
4. mcpClient.callTool({ name,     -> route tool_use to MCP server
         arguments })
5. mcpResult.isError?             -> check before sending tool_result
6. tool_result { is_error: true } -> inform Claude if call failed
```

### Error handling checklist

```
mcpResult.isError === true
  -> pass is_error: true in tool_result
  -> Claude will acknowledge the failure in its reply

isRetryable: false  -> do not retry; escalate or report
isRetryable: true   -> transient error; retry may succeed
```

### Authentication rules

```
✓ Credentials injected via env vars in .mcp.json (stdio) or headers (SSE)
✓ Server reads secrets with Deno.env.get() / process.env at startup
✗ Never accept secrets via tool arguments or request params
```

### MCP series index

| Notebook | Focus |
|---|---|
| `00_mcp_vs_tool_use` | Why MCP, architecture comparison |
| `01_mcp_primitives` | Tool / Resource / Prompt — when to use each |
| `02_mcp_server_basics` | Build a server, InMemoryTransport, .mcp.json |
| `03_tool_schema_design` | Fix broken schemas (high-frequency exam pattern) |
| **`04_mcp_client_with_claude`** | **End-to-end integration + error handling + auth** |